# Train balance bot using PPO with domain randomization

Run each cell by pressing `shift + enter`.

We redo the original training and add more phases to our curriculum learning. In addition to training the bot to stay upright (phase 1) and reduce movement off the origin (phase 2), we introduce increasingly difficult domain randomization schemes in phases 3, 4 and 5.

In [1]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

In [2]:
# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import the custom environment and PPO training module
from envs.balance_bot_env_dr import BalanceBotEnv, DomainRandomConfig
from rl.ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv, export_actor_onnx
from rl.onnx_actor_to_c import export_onnx_actor_to_c

In [3]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 4               # Number of parallel environments. Only the first will be rendered.
STEPS_PER_ENV = 500_000    # Number of simulation steps to perform per environment

## Configure PPO and environment

In [4]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    actor_hidden_layers = 2,       # Number of hidden layers in the actor network
    actor_hidden_size = 16,        # Number of nodes in each hidden layer in the actor
    critic_hidden_layers = 2,      # Number of hidden layers in the critic network
    critic_hidden_size = 16,       # Number of nodes in each hidden layer in the critic
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.000,              # Match MJCF opt.timestep for real-time rendering (or 0 for fast)
)

In [5]:
def make_balance_bot_env(render, **kwargs):
    """Function to create an environment for our balance bot"""
    # Create the environment and set the render mode
    env = BalanceBotEnv(
        mjcf_path    = MJCF_PATH,
        render_mode  = "human" if render else None,
        **kwargs
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs balance bot environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(
            lambda render=(i==0), kw=kwargs: make_balance_bot_env(render, **kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

## Phase 1: Balance only

We'll start with the easy task of only penalizing if the robot tilts (pitches) forward.

In [6]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-1"

# Create an environment that only rewards staying upright
envs = make_envs(
    NUM_ENVS,
    pitch_penalty_coef=0.5,
    action_penalty_coef=0.01,
    position_penalty_coef=0.0,
    yaw_penalty_coef=0.0
)

In [ ]:
# Choo choo train
result = train(ppo_config, envs=envs)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

## Phase 2: Penalize position and rotation

We start with the agent (actor and critic networks) trained in phase 1. We then update the existing environments to apply a penalty for any motion off the origin as well as rotation around the Z axis (yaw). We then train the existing agent further in this new environment.

In [7]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-2"

# Update the position and rotation coefficients in the existing environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.position_penalty_coef = 0.001
    env.yaw_penalty_coef = 0.1

In [ ]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

## Phase 3: Observation noise and action delay

With the robot staying upright and mostly in place, we add the additional difficulty of randomizing the observations (injecting noise) and introducing a random 0..1 timestep delay of how long it takes the chosen action to take effect.

The noisy observations model how real IMUs work. You will often find some noise in their readings, and it can help the robot deal with biases from an imperfectly calibrated IMU.

The action delay models the time it takes for the robot to respond to a chosen action. In simulation, a chosen action takes effect on the very next `step()` call. In real hardware, it could take up to a few milliseconds, thanks to extra code, communication bus latency (e.g. I2C), and motor lag.

In [8]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-3"

# Create data randomization config to add obs noise and action delay
dr = DomainRandomConfig(
    obs_noise_std_dev=0.01,   # Inject some noise into the observations
    action_delay_steps=1,     # 1 step (5ms) delay
    action_delay_random=True, # Vary 0-1 steps each episode
)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [9]:
# DEBUG: Uncomment this cell to load a previously saved model. Use this to skip training
# from previous phases. Note that you will still need to run the cells in each phase that update
# the experiment name and environment.

# from rl.ppo_trainer import Agent, TrainResult

# # Change this path to point to the saved model you want to use
# model_path = Path("runs/BalanceBot-v0__balance-bot-phase-2__42__1778853328/best_model.cleanrl_model")

# # Load agent from previous run
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# agent = Agent(envs, ppo_config).to(device)
# agent.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
# agent.eval()
# print(f"Loaded model from {model_path}")

# # Wrap agent in a dummy result
# result = TrainResult(
#     agent = agent,
#     checkpoint_dir = model_path.parent,
#     best_model_path = model_path,
#     final_model_path = None,
#     best_mean_return = 0,
# )

Loaded model from runs/BalanceBot-v0__balance-bot-phase-2__42__1778853328/best_model.cleanrl_model


In [10]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-3__42__1778858565
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-3
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-3__42__1778858565/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=147.09) to runs/BalanceBot-v0__balance-bot-phase-3__42__1778858565/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-3__42__1778858565/checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=167.36) to runs/BalanceBot-v0__balance-bot-phase-3__42__1778858565/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-3__42__1778858565/checkpoint_iter0150.cleanrl_model
New best model saved (mean_return=316.26) to runs/BalanceBot-v0__balance-bot-phase-3__42__1778858565/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-3__42__1778858565/checkpoint_iter0200.cleanrl_model
New best model saved (mean_return=1831.05) to

In [11]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Mean return: 209.90
Exported charts_metrics.csv (5 metrics, 7372 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 4

In [12]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-4"

# Add motor noise and pushes to our domain randomization config
dr.motor_noise_scale = 0.02 # +/-2% motor noise
dr.push_prob = 0.005        # 0.5% chance of push per step
dr.push_force_max_n = 0.3   # Gentle pushes

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [13]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-4__42__1778859480
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-4
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-4__42__1778859480/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=1199.52) to runs/BalanceBot-v0__balance-bot-phase-4__42__1778859480/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-4__42__1778859480/checkpoint_iter0100.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-4__42__1778859480/checkpoint_iter0150.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-4__42__1778859480/checkpoint_iter0200.cleanrl_model
New best model saved (mean_return=9652.30) to runs/BalanceBot-v0__balance-bot-phase-4__42__1778859480/best_model.cleanrl_model
Final model saved to runs/BalanceBot-v0__balance-bot-phase-4__42__1778859480/balance-bot-phase-4_final.cleanrl_model
Best model mean return: 9652.30, saved to runs/Balan

In [14]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Mean return: 2045.06
Exported charts_metrics.csv (5 metrics, 3312 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 5

In [15]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-5"

# Add mass and firction randomization to the DR config
dr.mass_scale_range= (0.8, 1.2)       # +/-20% mass variation
dr.friction_scale_range = (0.5, 1.5)  # 50-150% friction variation

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [16]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-5__42__1778860489
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-5
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-5__42__1778860489/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=6165.00) to runs/BalanceBot-v0__balance-bot-phase-5__42__1778860489/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-5__42__1778860489/checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=9939.74) to runs/BalanceBot-v0__balance-bot-phase-5__42__1778860489/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-5__42__1778860489/checkpoint_iter0150.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-5__42__1778860489/checkpoint_iter0200.cleanrl_model
New best model saved (mean_return=9958.20) to runs/BalanceBot-v0__balance-bot-phase-5__42__1778860489/best_model.cleanrl_model
Final model saved to runs/BalanceBot-v0__b

In [17]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Mean return: 2038.10
Exported charts_metrics.csv (5 metrics, 526 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Phase 6

In [18]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-6"

# Add motor gain randomization to our DR config
dr.motor_gain_range = (0.6, 1.0)    # 60% to 100% motor torque per episode

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [19]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-6__42__1778861421
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-6
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-6__42__1778861421/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=8177.08) to runs/BalanceBot-v0__balance-bot-phase-6__42__1778861421/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-6__42__1778861421/checkpoint_iter0100.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-6__42__1778861421/checkpoint_iter0150.cleanrl_model
New best model saved (mean_return=9097.59) to runs/BalanceBot-v0__balance-bot-phase-6__42__1778861421/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__balance-bot-phase-6__42__1778861421/checkpoint_iter0200.cleanrl_model
Final model saved to runs/BalanceBot-v0__balance-bot-phase-6__42__1778861421/balance-bot-phase-6_final.cleanrl_model
Best model mean return: 9097.59, saved to runs/Balan

In [20]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Mean return: 812.20
Exported charts_metrics.csv (5 metrics, 483 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


## Clean up and save model

At this point, we are done with training. We want to delete the environments and save the best actor from our final training phase. We'll export this actor network as an ONNX file that can be used on a variety of hardware platforms.

In [21]:
# Close the environments
for idx, env in enumerate(envs.envs):
    print(f"Closing env {idx}")
    env.env.close()

Closing env 0
Closing env 1
Closing env 2
Closing env 3


In [22]:
# Get observation and action sizes
obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]

# Export the actor network as an ONNX model
export_actor_onnx(
    model_path=result.best_model_path,
    output_path=result.checkpoint_dir / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)

/workspace/software/rl/ppo_trainer.py:381: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0515 17:43:58.814000 16301 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0515 17:43:58.816000 16301 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0515 17:43:58.818000 16301 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Actor exported to ONNX: runs/BalanceBot-v0__balance-bot-phase-6__42__1778861421/actor.onnx


/opt/pyenv/versions/3.12.13/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [23]:
# Export actor to .h file
export_onnx_actor_to_c(
    onnx_path   = result.checkpoint_dir / "actor.onnx",
    output_path = result.checkpoint_dir / "actor.h"
)

Weights found:
  0.weight: shape=(16, 4), dtype=float32
  0.bias: shape=(16,), dtype=float32
  2.weight: shape=(16, 16), dtype=float32
  2.bias: shape=(16,), dtype=float32
  4.weight: shape=(2, 16), dtype=float32
  4.bias: shape=(2,), dtype=float32
C header written to runs/BalanceBot-v0__balance-bot-phase-6__42__1778861421/actor.h
  Layers:  3
  Obs:     4
  Actions: 2


PosixPath('runs/BalanceBot-v0__balance-bot-phase-6__42__1778861421/actor.h')